# Training notebook

This should train the thing:

In [ ]:
# get dependencies
!pip install -q "pettingzoo[classic]" python-chess gymnasium pyyaml

# get stockfish
!apt-get update -qq
!apt-get install -y stockfish

!mkdir -p /opt/homebrew/bin && ln -sf /usr/games/stockfish /opt/homebrew/bin/stockfish

In [ ]:
!git clone https://github.com/Rishi-Jain-27/chess-marl
%cd chess-marl
%cd src

# Select the hybrid architecture (chess_mappo.py imports the architecture on line 13)
!sed -i 's/from model_architectures.cnn import/from model_architectures.hybrid import/' chess_mappo.py

!python chess_mappo.py hybrid_chess --train

This should get it to kaggle models:

**NOTE** Must change the info in here to align with what architecture we are training.

In [ ]:
# credentials
import os
from getpass import getpass
os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ').strip()
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key (id): ').strip()
!pip install -q kaggle

In [ ]:
# Create model and upload training results
import json

USERNAME = os.environ['KAGGLE_USERNAME']
MODEL_SLUG = 'chess-bot-hybrid'
RESULTS_DIR = '/content/chess-marl/src/runs'

# Create the empty model
model_meta = {
    "ownerSlug": USERNAME,
    "title": "Chess Bot Hybrid",
    "slug": MODEL_SLUG,
    "subtitle": "MAPPO hybrid CNN-transformer actor-critic chess bot",
    "isPrivate": False,
    "description": "Training results for hybrid MAPPO chess agent",
    "publishTime": "",
    "provenanceSources": ""
}
with open(os.path.join(RESULTS_DIR, 'model-metadata.json'), 'w') as f:
    json.dump(model_meta, f, indent=2)

!kaggle models create -p {RESULTS_DIR}

# Add the result files
instance_meta = {
    "ownerSlug": USERNAME,
    "modelSlug": MODEL_SLUG,
    "instanceSlug": "hybrid-run",
    "framework": "pyTorch",
    "overview": "Hybrid MAPPO training results",
    "usage": "python chess_mappo.py hybrid_chess --train",
    "licenseName": "MIT",
    "fineTunable": False,
    "trainingData": [],
    "files": [] # this means you upload every file
}
with open(os.path.join(RESULTS_DIR, 'model-instance-metadata.json'), 'w') as f:
    json.dump(instance_meta, f, indent=2)

!kaggle models instances create -p {RESULTS_DIR}
